# Génération d’un manifeste IIIF compatible TIFY et GitHub Pages

Ce notebook fait trois choses :

1. convertit les images TIFF du volume 1 en JPEG ;
2. place les JPEG dans le dossier publié par GitHub Pages : `site/images/volume_1/` ;
3. génère un manifeste IIIF 3.0 dans : `site/iiif/manifest_Frene_volume_1.json`.

L’objectif est que la page `site/Manuscrit/manuscrit.html` puisse charger TIFY avec :

```javascript
manifestUrl: '../iiif/manifest_Frene_volume_1.json'
```


In [1]:
from pathlib import Path
import json
from PIL import Image, ImageOps

print("Pillow :", Image.__version__)

Pillow : 12.2.0


## 1. Configuration des chemins

Le notebook peut être exécuté depuis le dossier du notebook ou depuis la racine du dépôt.  
La fonction ci-dessous retrouve automatiquement la racine du projet en cherchant les dossiers `data` et `site`.


In [2]:
def trouver_racine_projet(depart=None):
    """Retrouve la racine du dépôt Projet_Frene.

    La racine est identifiée par la présence simultanée des dossiers
    `data` et `site`.
    """
    depart = Path.cwd() if depart is None else Path(depart)
    depart = depart.resolve()

    candidats = [depart] + list(depart.parents)

    for candidat in candidats:
        if (candidat / "data").exists() and (candidat / "site").exists():
            return candidat

    raise FileNotFoundError(
        "Impossible de trouver la racine du projet. "
        "Lance le notebook depuis le dépôt Projet_Frene ou adapte RACINE_PROJET manuellement."
    )


RACINE_PROJET = trouver_racine_projet()

# Dossier source : TIFF originaux
DOSSIER_IMAGES_TIFF = RACINE_PROJET / "data" / "Frêne_volume_1" / "Images"

# Dossiers publiés par GitHub Pages, car ton workflow publie le dossier site/
DOSSIER_SITE = RACINE_PROJET / "site"
DOSSIER_IMAGES_JPG = DOSSIER_SITE / "images" / "volume_1"
DOSSIER_IIIF_SITE = DOSSIER_SITE / "iiif"

DOSSIER_IMAGES_JPG.mkdir(parents=True, exist_ok=True)
DOSSIER_IIIF_SITE.mkdir(parents=True, exist_ok=True)

FICHIER_MANIFEST = DOSSIER_IIIF_SITE / "manifest_Frene_volume_1.json"

# URL publique GitHub Pages.
# Important : le dossier `site` devient la racine publiée du site.
BASE_URL = "https://raphix93.github.io/Projet_Frene"

# URL publique des JPEG après publication GitHub Pages.
BASE_URL_IMAGES = f"{BASE_URL}/images/volume_1"

# URL publique du manifeste après publication GitHub Pages.
BASE_URL_IIIF = f"{BASE_URL}/iiif"

print("Racine projet :", RACINE_PROJET)
print("TIFF source   :", DOSSIER_IMAGES_TIFF)
print("JPEG sortie   :", DOSSIER_IMAGES_JPG)
print("IIIF sortie   :", DOSSIER_IIIF_SITE)
print("Manifest      :", FICHIER_MANIFEST)

Racine projet : C:\Users\rroll\Documents\GitHub\Projet_Frene
TIFF source   : C:\Users\rroll\Documents\GitHub\Projet_Frene\data\Frêne_volume_1\Images
JPEG sortie   : C:\Users\rroll\Documents\GitHub\Projet_Frene\site\images\volume_1
IIIF sortie   : C:\Users\rroll\Documents\GitHub\Projet_Frene\site\iiif
Manifest      : C:\Users\rroll\Documents\GitHub\Projet_Frene\site\iiif\manifest_Frene_volume_1.json


## 2. Métadonnées du manifeste

Ces informations apparaîtront dans TIFY.


In [3]:
TITRE = "Journal de Théophile Rémy Frêne, volume 1"
AUTEUR = "Théophile Rémy Frêne"
DATE = "1741"
INSTITUTION = "Office des archives de l'État de Neuchâtel"
COTE = "FRENE THEOPHILE-REMY"
LANGUE = "français"
LICENCE = "CC-BY 4.0"
ARK = "https://floraweb.ne.ch/flora/ark:/37964/001136"
DESCRIPTION = "Journal manuscrit de Théophile Rémy Frêne, pasteur neuchâtelois."
ATTRIBUTION = "Office des archives de l'État de Neuchâtel / Raphaël Rollinet" 

## 3. Conversion des TIFF en JPEG

TIFY s’affiche dans un navigateur. Les TIFF sont donc convertis en JPEG, format beaucoup mieux supporté par GitHub Pages et les navigateurs.

Par défaut :
- les dimensions originales sont conservées ;
- les JPEG sont enregistrés en qualité 85 ;
- le nom `Image00001.tif` devient `Image00001.jpg`.


In [4]:
def lister_tiff(dossier):
    """Liste les images TIFF du dossier source dans l’ordre alphabétique."""
    extensions = {".tif", ".tiff"}
    return sorted(
        fichier for fichier in dossier.iterdir()
        if fichier.is_file() and fichier.suffix.lower() in extensions
    )


def convertir_tiff_en_jpg(fichier_tiff, dossier_sortie, qualite=85, ecraser=False):
    """Convertit un fichier TIFF en JPEG compatible web.

    Les images sont converties en RGB afin d’éviter les problèmes liés
    aux modes couleurs ou à la transparence.
    """
    fichier_jpg = dossier_sortie / f"{fichier_tiff.stem}.jpg"

    if fichier_jpg.exists() and not ecraser:
        with Image.open(fichier_jpg) as image_existante:
            largeur, hauteur = image_existante.size
        return fichier_jpg, largeur, hauteur, "existe déjà"

    with Image.open(fichier_tiff) as image:
        image = ImageOps.exif_transpose(image)

        if image.mode not in ("RGB", "L"):
            image = image.convert("RGB")
        elif image.mode == "L":
            image = image.convert("RGB")

        largeur, hauteur = image.size

        image.save(
            fichier_jpg,
            "JPEG",
            quality=qualite,
            optimize=True,
            progressive=True
        )

    return fichier_jpg, largeur, hauteur, "créé"


images_tiff = lister_tiff(DOSSIER_IMAGES_TIFF)

if not images_tiff:
    raise FileNotFoundError(f"Aucune image TIFF trouvée dans : {DOSSIER_IMAGES_TIFF}")

images_jpg = []

for fichier_tiff in images_tiff:
    fichier_jpg, largeur, hauteur, statut = convertir_tiff_en_jpg(
        fichier_tiff,
        DOSSIER_IMAGES_JPG,
        qualite=85,
        ecraser=False
    )

    images_jpg.append({
        "fichier": fichier_jpg,
        "largeur": largeur,
        "hauteur": hauteur
    })

print(f"TIFF trouvés : {len(images_tiff)}")
print(f"JPEG disponibles : {len(images_jpg)}")
print("Exemple :", images_jpg[0]["fichier"])

TIFF trouvés : 45
JPEG disponibles : 45
Exemple : C:\Users\rroll\Documents\GitHub\Projet_Frene\site\images\volume_1\Image00001.jpg


## 4. Fonctions IIIF

Le manifeste pointe maintenant vers les JPEG publiés dans `site/images/volume_1/`.


In [5]:
def valeur_langue(texte, langue="fr"):
    """Retourne une valeur multilingue conforme à IIIF Presentation API 3."""
    return {langue: [texte]}


def creer_canvas(info_image, numero_page):
    """Crée un Canvas IIIF pour une image JPEG."""
    fichier_image = info_image["fichier"]
    largeur = info_image["largeur"]
    hauteur = info_image["hauteur"]

    nom_image = fichier_image.name
    image_url = f"{BASE_URL_IMAGES}/{nom_image}"

    canvas_id = f"{BASE_URL_IIIF}/canvas/volume1/p{numero_page}"
    annotation_page_id = f"{canvas_id}/annotation-page"
    annotation_id = f"{canvas_id}/annotation/image"

    return {
        "id": canvas_id,
        "type": "Canvas",
        "label": valeur_langue(f"Page {numero_page}"),
        "height": hauteur,
        "width": largeur,
        "items": [
            {
                "id": annotation_page_id,
                "type": "AnnotationPage",
                "items": [
                    {
                        "id": annotation_id,
                        "type": "Annotation",
                        "motivation": "painting",
                        "body": {
                            "id": image_url,
                            "type": "Image",
                            "format": "image/jpeg",
                            "height": hauteur,
                            "width": largeur
                        },
                        "target": canvas_id
                    }
                ]
            }
        ]
    }

## 5. Création du manifeste IIIF dans `site/iiif`

Après exécution, le manifeste sera accessible publiquement à l’adresse :

```text
https://raphix93.github.io/Projet_Frene/iiif/manifest_Frene_volume_1.json
```


In [6]:
manifest = {
    "@context": "http://iiif.io/api/presentation/3/context.json",
    "id": f"{BASE_URL_IIIF}/manifest_Frene_volume_1.json",
    "type": "Manifest",
    "label": valeur_langue(TITRE),
    "summary": valeur_langue(DESCRIPTION),
    "metadata": [
        {"label": valeur_langue("Auteur"), "value": valeur_langue(AUTEUR)},
        {"label": valeur_langue("Date"), "value": valeur_langue(DATE)},
        {"label": valeur_langue("Institution"), "value": valeur_langue(INSTITUTION)},
        {"label": valeur_langue("Cote"), "value": valeur_langue(COTE)},
        {"label": valeur_langue("Langue"), "value": valeur_langue(LANGUE)},
        {"label": valeur_langue("Licence"), "value": valeur_langue(LICENCE)},
        {"label": valeur_langue("ARK"), "value": valeur_langue(ARK)}
    ],
    "rights": "https://creativecommons.org/licenses/by/4.0/",
    "requiredStatement": {
        "label": valeur_langue("Attribution"),
        "value": valeur_langue(ATTRIBUTION)
    },
    "homepage": [
        {
            "id": ARK,
            "type": "Text",
            "label": valeur_langue("Notice d'origine")
        }
    ],
    "items": [
        creer_canvas(info_image, numero_page)
        for numero_page, info_image in enumerate(images_jpg, start=1)
    ]
}

with open(FICHIER_MANIFEST, "w", encoding="utf-8") as fichier:
    json.dump(manifest, fichier, ensure_ascii=False, indent=2)

print(f"Manifest créé : {FICHIER_MANIFEST}")
print(f"Nombre de canvas : {len(manifest['items'])}")
print("URL publique attendue :", f"{BASE_URL_IIIF}/manifest_Frene_volume_1.json")

Manifest créé : C:\Users\rroll\Documents\GitHub\Projet_Frene\site\iiif\manifest_Frene_volume_1.json
Nombre de canvas : 45
URL publique attendue : https://raphix93.github.io/Projet_Frene/iiif/manifest_Frene_volume_1.json


## 6. Vérification locale du manifeste

Cette cellule vérifie que :
- le manifeste contient bien des JPEG ;
- aucun TIFF n’est référencé ;
- les URLs correspondent à la racine GitHub Pages du dossier `site`.


In [7]:
with open(FICHIER_MANIFEST, "r", encoding="utf-8") as fichier:
    manifest_test = json.load(fichier)

urls_images = [
    canvas["items"][0]["items"][0]["body"]["id"]
    for canvas in manifest_test["items"]
]

formats_images = [
    canvas["items"][0]["items"][0]["body"]["format"]
    for canvas in manifest_test["items"]
]

print("Type :", manifest_test["type"])
print("Titre :", manifest_test["label"]["fr"][0])
print("Nombre de canvas :", len(manifest_test["items"]))
print("Première image :", urls_images[0])
print("Premier format :", formats_images[0])

assert manifest_test["id"] == "https://raphix93.github.io/Projet_Frene/iiif/manifest_Frene_volume_1.json"
assert all(url.startswith("https://raphix93.github.io/Projet_Frene/images/volume_1/") for url in urls_images)
assert all(url.endswith(".jpg") for url in urls_images)
assert all(format_image == "image/jpeg" for format_image in formats_images)

print("Vérification OK : le manifeste est prêt pour TIFY.")

Type : Manifest
Titre : Journal de Théophile Rémy Frêne, volume 1
Nombre de canvas : 45
Première image : https://raphix93.github.io/Projet_Frene/images/volume_1/Image00001.jpg
Premier format : image/jpeg
Vérification OK : le manifeste est prêt pour TIFY.


## 7. Bloc TIFY à placer dans `site/Manuscrit/manuscrit.html`

Depuis `site/Manuscrit/manuscrit.html`, le chemin relatif vers le manifeste est :

```text
../iiif/manifest_Frene_volume_1.json
```

Le bloc HTML/JavaScript à utiliser est donc :

```html
<div id="tify"></div>

<script type="module">
import Tify from 'https://cdn.jsdelivr.net/npm/tify@0.35.0/dist/tify.js'

new Tify({
    container: '#tify',
    manifestUrl: '../iiif/manifest_Frene_volume_1.json'
})
</script>
```

N’oublie pas aussi le CSS de TIFY dans le `<head>` :

```html
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/tify@0.35.0/dist/tify.css">

<style>
#tify {
    height: 80vh;
    width: 100%;
}
</style>
```
